In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.tallerspark;

In [0]:
BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

folders = [
    f"{BASE_PATH}/raw_jsonl/secop_ii_contratos/year=2025",
    f"{BASE_PATH}/raw_csv/secop_ii_contratos/year=2025",
    f"{BASE_PATH}/parquet/secop_ii_contratos/year=2025",
    f"{BASE_PATH}/manifest",
    f"{BASE_PATH}/logs"
]

for folder in folders:
    dbutils.fs.mkdirs(folder)
    print(f"Created: {folder}")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/tallerspark/secop"))

In [0]:
import requests
import json
import pandas as pd
from datetime import datetime, timezone
import time

YEAR = 2025

START_DATE = "2025-01-01T00:00:00"
END_DATE = "2026-01-01T00:00:00"

SOURCE_NAME = "secop_ii_contratos"
BASE_URL = "https://www.datos.gov.co/resource/jbjy-vk9h.json"
DATE_COLUMN = "fecha_de_firma"

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

RAW_JSONL_PATH = f"{BASE_PATH}/raw_jsonl/{SOURCE_NAME}/year={YEAR}"
RAW_CSV_PATH = f"{BASE_PATH}/raw_csv/{SOURCE_NAME}/year={YEAR}"
PARQUET_PATH = f"{BASE_PATH}/parquet/{SOURCE_NAME}/year={YEAR}"
MANIFEST_PATH = f"{BASE_PATH}/manifest"

BATCH_SIZE = 1000
MAX_BATCHES = 1

In [0]:
params_test = {
    "$limit": 5,
    "$offset": 0,
    "$where": f"{DATE_COLUMN} >= '{START_DATE}' AND {DATE_COLUMN} < '{END_DATE}'"
}

response = requests.get(BASE_URL, params=params_test, timeout=60)
response.raise_for_status()

sample_data = response.json()

print("Records downloaded:", len(sample_data))

df_sample = pd.DataFrame(sample_data)
display(df_sample)

In [0]:
def fetch_secop_batch(limit, offset, retries=3, sleep_seconds=5):
    params = {
        "$limit": limit,
        "$offset": offset,
        "$where": f"{DATE_COLUMN} >= '{START_DATE}' AND {DATE_COLUMN} < '{END_DATE}'",
        "$order": f"{DATE_COLUMN} ASC"
    }

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.json()

        except Exception as error:
            print(f"Attempt {attempt} failed at offset {offset}: {error}")

            if attempt == retries:
                raise

            time.sleep(sleep_seconds)

In [0]:
import requests
import json
import time
from datetime import datetime, timezone

YEAR = 2025

START_DATE = "2025-01-01T00:00:00"
END_DATE = "2026-01-01T00:00:00"

SOURCE_NAME = "secop_ii_contratos"
BASE_URL = "https://www.datos.gov.co/resource/jbjy-vk9h.json"
DATE_COLUMN = "fecha_de_firma"

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

RAW_JSONL_PATH = f"{BASE_PATH}/raw_jsonl/{SOURCE_NAME}/year={YEAR}"
MANIFEST_PATH = f"{BASE_PATH}/manifest"

BATCH_SIZE = 1000
MAX_BATCHES = 1


def fetch_secop_batch(limit, offset, retries=3, sleep_seconds=5):
    params = {
        "$limit": limit,
        "$offset": offset,
        "$where": f"{DATE_COLUMN} >= '{START_DATE}' AND {DATE_COLUMN} < '{END_DATE}'",
        "$order": f"{DATE_COLUMN} ASC"
    }

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.json()

        except Exception as error:
            print(f"Attempt {attempt} failed at offset {offset}: {error}")

            if attempt == retries:
                raise

            time.sleep(sleep_seconds)


manifest = []

for batch_number in range(MAX_BATCHES):
    offset = batch_number * BATCH_SIZE

    print(f"Downloading batch {batch_number} | offset={offset} | limit={BATCH_SIZE}")

    records = fetch_secop_batch(
        limit=BATCH_SIZE,
        offset=offset
    )

    print("Records received:", len(records))

    if not records:
        print("No more records available.")
        break

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

    jsonl_file = (
        f"{RAW_JSONL_PATH}/"
        f"batch_{batch_number:04d}_offset_{offset}_limit_{BATCH_SIZE}_{timestamp}.jsonl"
    )

    # Correct for Databricks Unity Catalog Volumes
    with open(jsonl_file, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    manifest.append({
        "source": SOURCE_NAME,
        "year": YEAR,
        "batch_number": batch_number,
        "offset": offset,
        "limit": BATCH_SIZE,
        "records": len(records),
        "jsonl_file": jsonl_file,
        "download_utc": datetime.now(timezone.utc).isoformat()
    })

    print("Saved JSONL:", jsonl_file)

In [0]:
df_raw = spark.read.json(RAW_JSONL_PATH)

print("Total records:", df_raw.count())
print("Total columns:", len(df_raw.columns))

display(df_raw.limit(10))

In [0]:
df_raw.printSchema()

In [0]:
RAW_CSV_PATH = "/Volumes/workspace/default/tallerspark/secop/raw_csv/secop_ii_contratos/year=2025/batch_0000_csv"

# Flatten struct columns for CSV export
df_flat = df_raw.withColumn("urlproceso_url", df_raw.urlproceso.url).drop("urlproceso")

(
    df_flat.write
    .mode("overwrite")
    .option("header", "true")
    .csv(RAW_CSV_PATH)
)

print("CSV saved in:", RAW_CSV_PATH)

In [0]:
PARQUET_PATH = "/Volumes/workspace/default/tallerspark/secop/parquet/secop_ii_contratos/year=2025/batch_0000_parquet"

(
    df_raw.write
    .mode("overwrite")
    .parquet(PARQUET_PATH)
)

print("Parquet saved in:", PARQUET_PATH)

In [0]:
import json
from datetime import datetime, timezone

MANIFEST_PATH = "/Volumes/workspace/default/tallerspark/secop/manifest/manifest_secop_2025_batch_0000.json"

manifest_data = {
    "source": "secop_ii_contratos",
    "year": 2025,
    "batch_number": 0,
    "offset": 0,
    "limit": 1000,
    "records": df_raw.count(),
    "raw_jsonl_path": RAW_JSONL_PATH,
    "raw_csv_path": RAW_CSV_PATH,
    "parquet_path": PARQUET_PATH,
    "created_utc": datetime.now(timezone.utc).isoformat()
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest_data, f, ensure_ascii=False, indent=4)

print("Manifest saved:", MANIFEST_PATH)

In [0]:
BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

display(dbutils.fs.ls(BASE_PATH))

In [0]:
import os
import requests
import json
import time
from datetime import datetime, timezone

YEAR = 2025
START_DATE = "2025-01-01T00:00:00"
END_DATE = "2026-01-01T00:00:00"

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

BATCH_SIZE = 1000
MAX_BATCHES = 1

SOURCES = {
    "secop_ii_contratos": {
        "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
        "date_column": "fecha_de_firma",
        "apply_year_filter": True
    },
    "secop_ii_adiciones": {
        "url": "https://www.datos.gov.co/resource/cb9c-h8sn.json",
        "date_column": "fecharegistro",
        "apply_year_filter": True
    },
    "secop_ii_ejecucion": {
        "url": "https://www.datos.gov.co/resource/mfmm-jqmq.json",
        "date_column": "fechacreacion",
        "apply_year_filter": True
    },
    "divipola_municipios": {
        "url": "https://www.datos.gov.co/resource/gdxc-w37w.json",
        "date_column": None,
        "apply_year_filter": False
    }
}

In [0]:
for source_name in SOURCES.keys():
    folders = [
        f"{BASE_PATH}/raw_jsonl/{source_name}/year={YEAR}",
        f"{BASE_PATH}/raw_csv/{source_name}/year={YEAR}",
        f"{BASE_PATH}/parquet/{source_name}/year={YEAR}"
    ]

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        print(f"Folder ready: {folder}")

os.makedirs(f"{BASE_PATH}/manifest", exist_ok=True)
os.makedirs(f"{BASE_PATH}/logs", exist_ok=True)

In [0]:
def build_params(source_config, limit, offset):
    params = {
        "$limit": limit,
        "$offset": offset
    }

    if source_config["apply_year_filter"]:
        date_column = source_config["date_column"]
        params["$where"] = f"{date_column} >= '{START_DATE}' AND {date_column} < '{END_DATE}'"
        params["$order"] = f"{date_column} ASC"

    return params


def fetch_batch(source_config, limit, offset, retries=3, sleep_seconds=5):
    params = build_params(source_config, limit, offset)

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(source_config["url"], params=params, timeout=120)
            response.raise_for_status()
            return response.json()

        except Exception as error:
            print(f"Attempt {attempt} failed at offset {offset}: {error}")

            if attempt == retries:
                raise

            time.sleep(sleep_seconds)


def download_source_layer0(source_name, batch_size=1000, max_batches=1):
    source_config = SOURCES[source_name]

    raw_jsonl_path = f"{BASE_PATH}/raw_jsonl/{source_name}/year={YEAR}"
    manifest = []

    for batch_number in range(max_batches):
        offset = batch_number * batch_size

        print(f"Downloading source={source_name} | batch={batch_number} | offset={offset} | limit={batch_size}")

        records = fetch_batch(source_config, batch_size, offset)

        print(f"Records received: {len(records)}")

        if not records:
            print("No more records available.")
            break

        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

        jsonl_file = (
            f"{raw_jsonl_path}/"
            f"batch_{batch_number:04d}_offset_{offset}_limit_{batch_size}_{timestamp}.jsonl"
        )

        with open(jsonl_file, "w", encoding="utf-8") as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        manifest.append({
            "source": source_name,
            "year": YEAR,
            "batch_number": batch_number,
            "offset": offset,
            "limit": batch_size,
            "records": len(records),
            "jsonl_file": jsonl_file,
            "download_utc": datetime.now(timezone.utc).isoformat()
        })

        print(f"Saved JSONL: {jsonl_file}")

    manifest_file = f"{BASE_PATH}/manifest/manifest_{source_name}_{YEAR}.json"

    with open(manifest_file, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=4)

    print(f"Manifest saved: {manifest_file}")

    return manifest

In [0]:
sources_to_download = [
    "secop_ii_adiciones",
    "secop_ii_ejecucion",
    "divipola_municipios"
]

all_manifests = {}

for source_name in sources_to_download:
    manifest = download_source_layer0(
        source_name=source_name,
        batch_size=1000,
        max_batches=1
    )

    all_manifests[source_name] = manifest

In [0]:
for source_name in SOURCES.keys():
    path = f"{BASE_PATH}/raw_jsonl/{source_name}/year={YEAR}"
    print("\nSOURCE:", source_name)
    display(dbutils.fs.ls(path))

In [0]:
for source_name in SOURCES.keys():
    raw_jsonl_path = f"{BASE_PATH}/raw_jsonl/{source_name}/year={YEAR}"
    raw_csv_path = f"{BASE_PATH}/raw_csv/{source_name}/year={YEAR}/batch_0000_csv"
    parquet_path = f"{BASE_PATH}/parquet/{source_name}/year={YEAR}/batch_0000_parquet"

    print(f"Processing source: {source_name}")

    df = spark.read.json(raw_jsonl_path)

    row_count = df.count()

    print(f"Rows: {row_count} | Columns: {len(df.columns)}")

    # Compute schema once after loading
    schema_fields = df.schema.fields

    # Flatten struct columns for CSV export
    # Build column expressions all at once to avoid nested plans
    columns_to_select = []
    for field in schema_fields:
        if str(field.dataType).startswith("StructType"):
            # Extract nested field and flatten
            if field.name == "urlproceso" and "url" in [f.name for f in field.dataType.fields]:
                columns_to_select.append(df[field.name].url.alias(f"{field.name}_url"))
        else:
            columns_to_select.append(df[field.name])

    df_for_csv = df.select(*columns_to_select)

    (
        df_for_csv.write
        .mode("overwrite")
        .option("header", "true")
        .csv(raw_csv_path)
    )

    (
        df.write
        .mode("overwrite")
        .parquet(parquet_path)
    )

    print(f"CSV saved: {raw_csv_path}")
    print(f"Parquet saved: {parquet_path}")

In [0]:
BATCH_SIZE = 50000
MAX_BATCHES = 2

sources_to_escalate = [
    "secop_ii_contratos",
    "secop_ii_adiciones",
    "secop_ii_ejecucion"
]

In [0]:
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print("Current run ID:", RUN_ID)

In [0]:
import os
import requests
import json
import time
from datetime import datetime, timezone

YEAR = 2025
START_DATE = "2025-01-01T00:00:00"
END_DATE = "2026-01-01T00:00:00"

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

BATCH_SIZE = 50000
MAX_BATCHES = 2

SOURCES = {
    "secop_ii_contratos": {
        "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
        "date_column": "fecha_de_firma",
        "apply_year_filter": True
    },
    "secop_ii_adiciones": {
        "url": "https://www.datos.gov.co/resource/cb9c-h8sn.json",
        "date_column": "fecharegistro",
        "apply_year_filter": True
    },
    "secop_ii_ejecucion": {
        "url": "https://www.datos.gov.co/resource/mfmm-jqmq.json",
        "date_column": "fechacreacion",
        "apply_year_filter": True
    }
}

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print("Run ID:", RUN_ID)


def build_params(source_config, limit, offset):
    params = {
        "$limit": limit,
        "$offset": offset
    }

    if source_config["apply_year_filter"]:
        date_column = source_config["date_column"]
        params["$where"] = f"{date_column} >= '{START_DATE}' AND {date_column} < '{END_DATE}'"
        params["$order"] = f"{date_column} ASC"

    return params


def fetch_batch(source_config, limit, offset, retries=3, sleep_seconds=10):
    params = build_params(source_config, limit, offset)

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(source_config["url"], params=params, timeout=180)
            response.raise_for_status()

            data = response.json()

            if isinstance(data, dict) and "error" in data:
                raise Exception(data)

            return data

        except Exception as error:
            print(f"Attempt {attempt} failed | offset={offset} | error={error}")

            if attempt == retries:
                raise

            time.sleep(sleep_seconds)


def download_source_100k(source_name):
    source_config = SOURCES[source_name]

    raw_jsonl_path = (
        f"{BASE_PATH}/raw_jsonl/{source_name}/year={YEAR}/run_id={RUN_ID}"
    )

    os.makedirs(raw_jsonl_path, exist_ok=True)

    manifest = []

    total_records = 0

    for batch_number in range(MAX_BATCHES):
        offset = batch_number * BATCH_SIZE

        print("=" * 80)
        print(f"Downloading source={source_name}")
        print(f"Batch={batch_number} | Offset={offset} | Limit={BATCH_SIZE}")

        records = fetch_batch(
            source_config=source_config,
            limit=BATCH_SIZE,
            offset=offset
        )

        received = len(records)
        total_records += received

        print(f"Records received: {received}")

        if received == 0:
            print("No more records available.")
            break

        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

        jsonl_file = (
            f"{raw_jsonl_path}/"
            f"batch_{batch_number:04d}_offset_{offset}_limit_{BATCH_SIZE}_{timestamp}.jsonl"
        )

        with open(jsonl_file, "w", encoding="utf-8") as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        manifest.append({
            "source": source_name,
            "year": YEAR,
            "run_id": RUN_ID,
            "batch_number": batch_number,
            "offset": offset,
            "limit": BATCH_SIZE,
            "records": received,
            "jsonl_file": jsonl_file,
            "download_utc": datetime.now(timezone.utc).isoformat()
        })

        print(f"Saved JSONL: {jsonl_file}")

        if received < BATCH_SIZE:
            print("Last available batch reached.")
            break

    manifest_file = f"{BASE_PATH}/manifest/manifest_{source_name}_{YEAR}_run_{RUN_ID}.json"

    with open(manifest_file, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=4)

    print("=" * 80)
    print(f"Finished source: {source_name}")
    print(f"Total records downloaded: {total_records}")
    print(f"Manifest saved: {manifest_file}")

    return manifest

In [0]:
all_manifests_100k = {}

for source_name in sources_to_escalate:
    all_manifests_100k[source_name] = download_source_100k(source_name)

In [0]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

# ============================================================
# FINAL QUERY RULE
# From 2025-01-01 to the real download datetime
# ============================================================

DOWNLOAD_TZ = ZoneInfo("America/Bogota")

DOWNLOAD_DATETIME = datetime.now(DOWNLOAD_TZ)
DOWNLOAD_DATE = DOWNLOAD_DATETIME.strftime("%Y-%m-%d")
DOWNLOAD_TIMESTAMP = DOWNLOAD_DATETIME.strftime("%Y-%m-%dT%H:%M:%S")

START_DATE = "2025-01-01T00:00:00"
END_DATE = DOWNLOAD_TIMESTAMP

PERIOD_NAME = f"from_2025_01_01_to_{DOWNLOAD_DATE}"
RUN_ID_FULL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print("Start date:", START_DATE)
print("Download end date:", END_DATE)
print("Period:", PERIOD_NAME)
print("Run ID:", RUN_ID_FULL)

In [0]:
def build_params(source_config, limit, offset):
    params = {
        "$limit": limit,
        "$offset": offset
    }

    if source_config["apply_date_filter"]:
        date_column = source_config["date_column"]

        params["$where"] = (
            f"{date_column} >= '{START_DATE}' "
            f"AND {date_column} <= '{END_DATE}'"
        )

        params["$order"] = f"{date_column} ASC"

    return params

In [0]:
SOURCES_FULL = {
    "secop_ii_contratos": {
        "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
        "date_column": "fecha_de_firma",
        "apply_date_filter": True
    },
    "secop_ii_adiciones": {
        "url": "https://www.datos.gov.co/resource/cb9c-h8sn.json",
        "date_column": "fecharegistro",
        "apply_date_filter": True
    },
    "secop_ii_ejecucion": {
        "url": "https://www.datos.gov.co/resource/mfmm-jqmq.json",
        "date_column": "fechacreacion",
        "apply_date_filter": True
    },
    "divipola_municipios": {
        "url": "https://www.datos.gov.co/resource/gdxc-w37w.json",
        "date_column": None,
        "apply_date_filter": False
    }
}

In [0]:
raw_jsonl_path = (
    f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
)

In [0]:
def download_all_source(source_name):
    source_config = SOURCES_FULL[source_name]

    raw_jsonl_path = (
        f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    os.makedirs(raw_jsonl_path, exist_ok=True)

    manifest = []
    total_records = 0
    batch_number = 0

    while True:
        offset = batch_number * BATCH_SIZE

        print("=" * 100)
        print(f"Source: {source_name}")
        print(f"Batch: {batch_number}")
        print(f"Offset: {offset}")
        print(f"Limit: {BATCH_SIZE}")
        print(f"Query window: {START_DATE} to {END_DATE}")

        records = fetch_batch(
            source_config=source_config,
            limit=BATCH_SIZE,
            offset=offset
        )

        received = len(records)
        total_records += received

        print(f"Records received: {received}")

        if received == 0:
            print("No more records available.")
            break

        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

        jsonl_file = (
            f"{raw_jsonl_path}/"
            f"batch_{batch_number:04d}_"
            f"offset_{offset}_"
            f"limit_{BATCH_SIZE}_"
            f"{timestamp}.jsonl"
        )

        with open(jsonl_file, "w", encoding="utf-8") as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        manifest.append({
            "source": source_name,
            "period": PERIOD_NAME,
            "start_date": START_DATE,
            "download_end_date": END_DATE,
            "run_id": RUN_ID_FULL,
            "batch_number": batch_number,
            "offset": offset,
            "limit": BATCH_SIZE,
            "records": received,
            "jsonl_file": jsonl_file,
            "download_utc": datetime.now(timezone.utc).isoformat()
        })

        print(f"Saved JSONL: {jsonl_file}")

        if received < BATCH_SIZE:
            print("Last batch reached.")
            break

        batch_number += 1
        time.sleep(3)

    manifest_file = (
        f"{BASE_PATH}/manifest/"
        f"manifest_{source_name}_{PERIOD_NAME}_run_{RUN_ID_FULL}.json"
    )

    with open(manifest_file, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=4)

    print("=" * 100)
    print(f"Finished source: {source_name}")
    print(f"Total records downloaded: {total_records}")
    print(f"Manifest saved: {manifest_file}")

    return {
        "source": source_name,
        "period": PERIOD_NAME,
        "start_date": START_DATE,
        "download_end_date": END_DATE,
        "total_records": total_records,
        "batches": len(manifest),
        "manifest_file": manifest_file,
        "run_id": RUN_ID_FULL
    }

In [0]:
# ============================================================
# FUENTES A PROCESAR EN LA DESCARGA FINAL
# ============================================================

sources_to_download_full = [
    "secop_ii_contratos",
    "secop_ii_adiciones",
    "secop_ii_ejecucion",
    "divipola_municipios"
]

print("Sources ready:")
for source in sources_to_download_full:
    print("-", source)

In [0]:
print("BASE_PATH:", BASE_PATH)
print("PERIOD_NAME:", PERIOD_NAME)
print("RUN_ID_FULL:", RUN_ID_FULL)

In [0]:
import os
import requests
import json
import time
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

DOWNLOAD_TZ = ZoneInfo("America/Bogota")

DOWNLOAD_DATETIME = datetime.now(DOWNLOAD_TZ)
DOWNLOAD_DATE = DOWNLOAD_DATETIME.strftime("%Y-%m-%d")
DOWNLOAD_TIMESTAMP = DOWNLOAD_DATETIME.strftime("%Y-%m-%dT%H:%M:%S")

START_DATE = "2025-01-01T00:00:00"
END_DATE = DOWNLOAD_TIMESTAMP

PERIOD_NAME = f"from_2025_01_01_to_{DOWNLOAD_DATE}"
RUN_ID_FULL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"
BATCH_SIZE = 50000

print("Start date:", START_DATE)
print("End date:", END_DATE)
print("Period:", PERIOD_NAME)
print("Run ID:", RUN_ID_FULL)

In [0]:
# ============================================================
# SECOP BIG DATA - LAYER 0 FULL DOWNLOAD
# Final rule:
# From 2025-01-01 to the real download datetime
# ============================================================

import os
import json
import time
import requests
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

# ============================================================
# 1. CREATE / ENSURE UNITY CATALOG VOLUME
# ============================================================

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.tallerspark")

# ============================================================
# 2. GENERAL CONFIGURATION
# ============================================================

DOWNLOAD_TZ = ZoneInfo("America/Bogota")

DOWNLOAD_DATETIME = datetime.now(DOWNLOAD_TZ)
DOWNLOAD_DATE = DOWNLOAD_DATETIME.strftime("%Y-%m-%d")
DOWNLOAD_TIMESTAMP = DOWNLOAD_DATETIME.strftime("%Y-%m-%dT%H:%M:%S")

START_DATE = "2025-01-01T00:00:00"
END_DATE = DOWNLOAD_TIMESTAMP

PERIOD_NAME = f"from_2025_01_01_to_{DOWNLOAD_DATE}"
RUN_ID_FULL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

BATCH_SIZE = 50000

WRITE_PARQUET = True
WRITE_CSV = True

print("=" * 100)
print("SECOP BIG DATA - FINAL LAYER 0 DOWNLOAD")
print("=" * 100)
print("Start date:", START_DATE)
print("End date:", END_DATE)
print("Period:", PERIOD_NAME)
print("Run ID:", RUN_ID_FULL)
print("Base path:", BASE_PATH)
print("Batch size:", BATCH_SIZE)

# ============================================================
# 3. DATA SOURCES
# ============================================================

SOURCES_FULL = {
    "secop_ii_contratos": {
        "url": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
        "date_column": "fecha_de_firma",
        "apply_date_filter": True
    },
    "secop_ii_adiciones": {
        "url": "https://www.datos.gov.co/resource/cb9c-h8sn.json",
        "date_column": "fecharegistro",
        "apply_date_filter": True
    },
    "secop_ii_ejecucion": {
        "url": "https://www.datos.gov.co/resource/mfmm-jqmq.json",
        "date_column": "fechacreacion",
        "apply_date_filter": True
    },
    "divipola_municipios": {
        "url": "https://www.datos.gov.co/resource/gdxc-w37w.json",
        "date_column": None,
        "apply_date_filter": False
    }
}

sources_to_download_full = list(SOURCES_FULL.keys())

# ============================================================
# 4. CREATE FOLDER STRUCTURE
# ============================================================

base_folders = [
    f"{BASE_PATH}/raw_jsonl",
    f"{BASE_PATH}/raw_csv",
    f"{BASE_PATH}/parquet",
    f"{BASE_PATH}/manifest",
    f"{BASE_PATH}/logs"
]

for folder in base_folders:
    os.makedirs(folder, exist_ok=True)

for source_name in sources_to_download_full:
    os.makedirs(f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}", exist_ok=True)
    os.makedirs(f"{BASE_PATH}/raw_csv/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}", exist_ok=True)
    os.makedirs(f"{BASE_PATH}/parquet/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}", exist_ok=True)

print("\nFolder structure created.")

# ============================================================
# 5. API PARAMETER BUILDER
# ============================================================

def build_params(source_config, limit, offset):
    params = {
        "$limit": limit,
        "$offset": offset
    }

    if source_config["apply_date_filter"]:
        date_column = source_config["date_column"]

        params["$where"] = (
            f"{date_column} >= '{START_DATE}' "
            f"AND {date_column} <= '{END_DATE}'"
        )

        params["$order"] = f"{date_column} ASC"

    return params

# ============================================================
# 6. API REQUEST WITH RETRIES
# ============================================================

def fetch_batch(source_config, limit, offset, retries=3, sleep_seconds=15):
    params = build_params(source_config, limit, offset)

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(
                source_config["url"],
                params=params,
                timeout=240
            )

            response.raise_for_status()
            data = response.json()

            if isinstance(data, dict) and "error" in data:
                raise Exception(data)

            return data

        except Exception as error:
            print(f"Attempt {attempt} failed | offset={offset}")
            print("Error:", error)

            if attempt == retries:
                raise

            print(f"Waiting {sleep_seconds} seconds before retry...")
            time.sleep(sleep_seconds)

# ============================================================
# 7. DOWNLOAD COMPLETE SOURCE
# ============================================================

def download_all_source(source_name):
    source_config = SOURCES_FULL[source_name]

    raw_jsonl_path = (
        f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    os.makedirs(raw_jsonl_path, exist_ok=True)

    manifest = []
    total_records = 0
    batch_number = 0

    while True:
        offset = batch_number * BATCH_SIZE

        print("\n" + "=" * 100)
        print(f"Downloading source: {source_name}")
        print(f"Batch: {batch_number}")
        print(f"Offset: {offset}")
        print(f"Limit: {BATCH_SIZE}")
        print(f"Query window: {START_DATE} to {END_DATE}")

        records = fetch_batch(
            source_config=source_config,
            limit=BATCH_SIZE,
            offset=offset
        )

        received = len(records)
        total_records += received

        print(f"Records received: {received}")

        if received == 0:
            print("No more records available.")
            break

        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

        jsonl_file = (
            f"{raw_jsonl_path}/"
            f"batch_{batch_number:04d}_"
            f"offset_{offset}_"
            f"limit_{BATCH_SIZE}_"
            f"{timestamp}.jsonl"
        )

        with open(jsonl_file, "w", encoding="utf-8") as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        manifest.append({
            "source": source_name,
            "period": PERIOD_NAME,
            "start_date": START_DATE,
            "download_end_date": END_DATE,
            "run_id": RUN_ID_FULL,
            "batch_number": batch_number,
            "offset": offset,
            "limit": BATCH_SIZE,
            "records": received,
            "jsonl_file": jsonl_file,
            "download_utc": datetime.now(timezone.utc).isoformat()
        })

        print(f"Saved JSONL: {jsonl_file}")

        if received < BATCH_SIZE:
            print("Last batch reached because received records are lower than batch size.")
            break

        batch_number += 1

        # Small pause to reduce API pressure
        time.sleep(3)

    manifest_file = (
        f"{BASE_PATH}/manifest/"
        f"manifest_{source_name}_{PERIOD_NAME}_run_{RUN_ID_FULL}.json"
    )

    with open(manifest_file, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=4)

    print("\n" + "=" * 100)
    print(f"Finished source: {source_name}")
    print(f"Total records downloaded: {total_records}")
    print(f"Manifest saved: {manifest_file}")

    return {
        "source": source_name,
        "period": PERIOD_NAME,
        "start_date": START_DATE,
        "download_end_date": END_DATE,
        "total_records": total_records,
        "batches": len(manifest),
        "manifest_file": manifest_file,
        "run_id": RUN_ID_FULL
    }

# ============================================================
# 8. DOWNLOAD ALL SOURCES
# ============================================================

full_download_summary = []

for source_name in sources_to_download_full:
    result = download_all_source(source_name)
    full_download_summary.append(result)

print("\n" + "=" * 100)
print("DOWNLOAD SUMMARY")
print("=" * 100)

for item in full_download_summary:
    print(item)

# ============================================================
# 9. VALIDATE RAW JSONL DOWNLOAD
# ============================================================

validation_summary = []

for source_name in sources_to_download_full:
    raw_jsonl_path = (
        f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    print("\n" + "=" * 100)
    print(f"Validating raw JSONL source: {source_name}")
    print("Path:", raw_jsonl_path)

    try:
        files = dbutils.fs.ls(raw_jsonl_path)
        print(f"Files found: {len(files)}")

        if len(files) == 0:
            print("No files found.")
            row_count = 0
            col_count = 0
        else:
            df_raw = spark.read.json(raw_jsonl_path)
            row_count = df_raw.count()
            col_count = len(df_raw.columns)

            print(f"Rows: {row_count}")
            print(f"Columns: {col_count}")

        validation_summary.append({
            "source": source_name,
            "raw_jsonl_path": raw_jsonl_path,
            "files": len(files),
            "rows": row_count,
            "columns": col_count
        })

    except Exception as error:
        print(f"Validation error for {source_name}: {error}")
        validation_summary.append({
            "source": source_name,
            "raw_jsonl_path": raw_jsonl_path,
            "files": 0,
            "rows": 0,
            "columns": 0,
            "error": str(error)
        })

# ============================================================
# 10. CONVERT TO PARQUET AND CSV
# ============================================================

for source_name in sources_to_download_full:
    raw_jsonl_path = (
        f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    parquet_path = (
        f"{BASE_PATH}/parquet/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    csv_path = (
        f"{BASE_PATH}/raw_csv/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"
    )

    print("\n" + "=" * 100)
    print(f"Converting source: {source_name}")
    print("Raw JSONL path:", raw_jsonl_path)

    try:
        df = spark.read.json(raw_jsonl_path)

        rows = df.count()
        cols = len(df.columns)

        print(f"Rows to convert: {rows}")
        print(f"Columns to convert: {cols}")

        if rows == 0:
            print("Skipping conversion because there are no records.")
            continue

        if WRITE_PARQUET:
            (
                df.write
                .mode("overwrite")
                .parquet(parquet_path)
            )
            print(f"Parquet saved: {parquet_path}")

        if WRITE_CSV:
            (
                df.write
                .mode("overwrite")
                .option("header", "true")
                .csv(csv_path)
            )
            print(f"CSV saved: {csv_path}")

    except Exception as error:
        print(f"Conversion error for {source_name}: {error}")

# ============================================================
# 11. SAVE GLOBAL EXECUTION SUMMARY
# ============================================================

global_summary = {
    "project": "SECOP Big Data 2025",
    "layer": "Layer 0",
    "period": PERIOD_NAME,
    "start_date": START_DATE,
    "download_end_date": END_DATE,
    "run_id": RUN_ID_FULL,
    "batch_size": BATCH_SIZE,
    "sources": full_download_summary,
    "validation": validation_summary,
    "created_utc": datetime.now(timezone.utc).isoformat()
}

global_summary_file = (
    f"{BASE_PATH}/manifest/"
    f"global_summary_{PERIOD_NAME}_run_{RUN_ID_FULL}.json"
)

with open(global_summary_file, "w", encoding="utf-8") as f:
    json.dump(global_summary, f, ensure_ascii=False, indent=4)

print("\n" + "=" * 100)
print("GLOBAL SUMMARY SAVED")
print("=" * 100)
print(global_summary_file)

# ============================================================
# 12. FINAL DISPLAY
# ============================================================

print("\n" + "=" * 100)
print("FINAL RESULT")
print("=" * 100)
print("Period:", PERIOD_NAME)
print("Run ID:", RUN_ID_FULL)
print("Base path:", BASE_PATH)
print("Raw JSONL:", f"{BASE_PATH}/raw_jsonl")
print("Parquet:", f"{BASE_PATH}/parquet")
print("CSV:", f"{BASE_PATH}/raw_csv")
print("Manifest:", f"{BASE_PATH}/manifest")

display(full_download_summary)
display(validation_summary)

In [0]:
BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

PERIOD_NAME = "from_2025_01_01_to_2026-06-02"  # replace if different
RUN_ID_FULL = "20260603T021642Z"               # replace with your real run_id

sources_to_download_full = [
    "secop_ii_contratos",
    "secop_ii_adiciones",
    "secop_ii_ejecucion",
    "divipola_municipios"
]

for source_name in sources_to_download_full:
    path = f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"

    print("=" * 80)
    print("SOURCE:", source_name)
    print("PATH:", path)

    try:
        files = dbutils.fs.ls(path)
        print("Files found:", len(files))
        display(files)

        if len(files) > 0:
            df = spark.read.json(path)
            print("Rows:", df.count())
            print("Columns:", len(df.columns))

    except Exception as e:
        print("No data found or error:", e)

In [0]:
BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

sources = [
    "secop_ii_contratos",
    "secop_ii_adiciones",
    "secop_ii_ejecucion",
    "divipola_municipios"
]

def safe_ls(path):
    try:
        return dbutils.fs.ls(path)
    except Exception as e:
        print(f"No existe o no se puede leer: {path}")
        return []

print("=" * 100)
print("BUSCANDO DATOS DESCARGADOS EN raw_jsonl")
print("=" * 100)

for source in sources:
    source_root = f"{BASE_PATH}/raw_jsonl/{source}"
    print("\n" + "=" * 100)
    print("SOURCE:", source)
    print("ROOT:", source_root)

    level_1 = safe_ls(source_root)

    for item_1 in level_1:
        print("  ", item_1.path)

        level_2 = safe_ls(item_1.path)

        for item_2 in level_2:
            print("      ", item_2.path)

            level_3 = safe_ls(item_2.path)

            jsonl_files = [
                f for f in level_3
                if f.path.endswith(".jsonl")
            ]

            if jsonl_files:
                print("          JSONL files found:", len(jsonl_files))
                for file in jsonl_files[:5]:
                    print("          -", file.name)

In [0]:
BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

PERIOD_NAME = "from_2025_01_01_to_2026-06-02"
RUN_ID_FULL = "20260603T031107Z"

sources_to_validate = [
    "secop_ii_contratos",
    "secop_ii_adiciones",
    "secop_ii_ejecucion",
    "divipola_municipios"
]

validation_results = []

for source_name in sources_to_validate:
    path = f"{BASE_PATH}/raw_jsonl/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"

    print("=" * 100)
    print("SOURCE:", source_name)
    print("PATH:", path)

    try:
        files = dbutils.fs.ls(path)
        jsonl_files = [f for f in files if f.path.endswith(".jsonl")]

        print("JSONL files:", len(jsonl_files))

        if len(jsonl_files) > 0:
            df = spark.read.json(path)

            rows = df.count()
            columns = len(df.columns)

            print("Rows:", rows)
            print("Columns:", columns)

            validation_results.append({
                "source": source_name,
                "path": path,
                "jsonl_files": len(jsonl_files),
                "rows": rows,
                "columns": columns
            })
        else:
            print("No JSONL files found.")

            validation_results.append({
                "source": source_name,
                "path": path,
                "jsonl_files": 0,
                "rows": 0,
                "columns": 0
            })

    except Exception as e:
        print("No data found or error:", e)

        validation_results.append({
            "source": source_name,
            "path": path,
            "jsonl_files": 0,
            "rows": 0,
            "columns": 0,
            "error": str(e)
        })

display(validation_results)

In [0]:
for item in validation_results:
    source_name = item["source"]

    if item["rows"] > 0:
        raw_path = item["path"]
        parquet_path = f"{BASE_PATH}/parquet/{source_name}/period={PERIOD_NAME}/run_id={RUN_ID_FULL}"

        df = spark.read.json(raw_path)

        (
            df.write
            .mode("overwrite")
            .parquet(parquet_path)
        )

        print(f"Parquet saved for {source_name}: {parquet_path}")